In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

## LOAD DATA

In [ ]:
sentiment = pd.read_csv('./csv_files/fear_greed_index.csv')
trades = pd.read_csv('./csv_files/historical_data.csv')
print("Sentiment dataset shape:", sentiment.shape)
print("Trades dataset shape:", trades.shape)

Sentiment dataset shape: (2644, 4)
Trades dataset shape: (13979, 16)


## CLEAN TRADES DATA

In [ ]:
# Convert IST timestamp string to datetime
trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'], errors='coerce')

In [ ]:
# Convert UNIX timestamp column to datetime
# (Dividing by 1000 only if it's in milliseconds — check range)
if trades['Timestamp'].max() > 1e12:
    trades['Timestamp'] = pd.to_datetime(trades['Timestamp'], unit='ms', errors='coerce')
else:
    trades['Timestamp'] = pd.to_datetime(trades['Timestamp'], unit='s', errors='coerce')

In [ ]:
# Extract DATE column
trades['date'] = trades['Timestamp'].dt.date

# Convert numeric fields
numeric_cols = [
    'Execution Price', 'Size Tokens', 'Size USD',
    'Start Position', 'Closed PnL', 'Fee'
]

for col in numeric_cols:
    trades[col] = pd.to_numeric(trades[col], errors='coerce')

# Clean side column (BUY/SELL to lowercase)
trades['Side'] = trades['Side'].astype(str).str.lower().str.strip()

print("\nTrades after cleaning:")
print(trades.head())


Trades after cleaning:
                                      Account  Coin  Execution Price  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9894   

   Size Tokens  Size USD Side       Timestamp IST  Start Position Direction  \
0       986.87   7872.16  buy 2024-02-12 22:50:00        0.000000       Buy   
1        16.00    127.68  buy 2024-02-12 22:50:00      986.524596       Buy   
2       144.09   1150.63  buy 2024-02-12 22:50:00     1002.518996       Buy   
3       142.98   1142.04  buy 2024-02-12 22:50:00     1146.558564       Buy   
4         8.73     69.75  buy 2024-02-12 22:50:00     1289.488521       Buy   

   Closed PnL                                   Transaction Hash  \


## CLEAN SENTIMENT DATA

In [ ]:
sentiment['timestamp'] = pd.to_datetime(sentiment['timestamp'], unit='s', errors='coerce')
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

sentiment['classification'] = sentiment['classification'].astype(str).str.lower().str.strip()
sentiment.rename(columns={'classification': 'sentiment'}, inplace=True)

print("\nClean Sentiment Sample:")
print(sentiment.head())


Clean Sentiment Sample:
            timestamp  value     sentiment        date
0 2018-02-01 05:30:00     30          fear  2018-02-01
1 2018-02-02 05:30:00     15  extreme fear  2018-02-02
2 2018-02-03 05:30:00     40          fear  2018-02-03
3 2018-02-04 05:30:00     24  extreme fear  2018-02-04
4 2018-02-05 05:30:00     11  extreme fear  2018-02-05


## MERGE BOTH DATASETS INTO A SINGLE DATAFRAME

In [ ]:
df = trades.merge(
    sentiment[['date', 'sentiment', 'value']],
    on='date',
    how='left'
)

print("\nMerged Data Sample:")
print(df[['Timestamp', 'Account', 'Coin', 'Execution Price', 'Size Tokens', 'Side', 'sentiment']].head())



Merged Data Sample:
            Timestamp                                     Account  Coin  \
0 2024-10-27 03:33:20  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107   
1 2024-10-27 03:33:20  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107   
2 2024-10-27 03:33:20  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107   
3 2024-10-27 03:33:20  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107   
4 2024-10-27 03:33:20  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107   

   Execution Price  Size Tokens Side sentiment  
0           7.9769       986.87  buy     greed  
1           7.9800        16.00  buy     greed  
2           7.9855       144.09  buy     greed  
3           7.9874       142.98  buy     greed  
4           7.9894         8.73  buy     greed  


In [ ]:
df['sentiment']

,sentiment
0,greed
1,greed
2,greed
3,greed
4,greed
...,...
13974,NaN
13975,NaN
13976,NaN
13977,NaN


## HANDLE MISSING SENTIMENT ROWS

In [ ]:
df['sentiment'] = df['sentiment'].fillna(method='ffill')
df['value'] = df['value'].fillna(method='ffill')

print("\nSentiment Value Counts After Fill:")
print(df['sentiment'].value_counts())


Sentiment Value Counts After Fill:
sentiment
fear     13147
greed      832
Name: count, dtype: int64


/tmp/ipython-input-3861443624.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['sentiment'] = df['sentiment'].fillna(method='ffill')
/tmp/ipython-input-3861443624.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['value'] = df['value'].fillna(method='ffill')


In [ ]:
print("Merged dataframe sample:")
df.head()

Merged dataframe sample:


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date,sentiment,value
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,buy,2024-02-12 22:50:00,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,5.201771e+10,True,0.345404,8.950000e+14,2024-10-27 03:33:20,2024-10-27,greed,74.0
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,buy,2024-02-12 22:50:00,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,5.201771e+10,True,0.005600,4.430000e+14,2024-10-27 03:33:20,2024-10-27,greed,74.0
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,buy,2024-02-12 22:50:00,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,5.201771e+10,True,0.050431,6.600000e+14,2024-10-27 03:33:20,2024-10-27,greed,74.0
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,buy,2024-02-12 22:50:00,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,5.201771e+10,True,0.050043,1.080000e+15,2024-10-27 03:33:20,2024-10-27,greed,74.0
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,buy,2024-02-12 22:50:00,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,5.201771e+10,True,0.003055,1.050000e+15,2024-10-27 03:33:20,2024-10-27,greed,74.0


## Feature Engineering

In [ ]:
df['closed_pnl'] = pd.to_numeric(df['Closed PnL'], errors='coerce')
df['win'] = df['closed_pnl'] > 0
df['loss'] = df['closed_pnl'] < 0
df['hour'] = df['Timestamp'].dt.hour
df['start_position'] = pd.to_numeric(df['Start Position'], errors='coerce')

# per-day aggregates (we will compute shortly)
df['trade_count'] = 1

# relative trade size = Size USD / (abs(Start Position) + eps)
eps = 1e-8
df['rel_trade_size'] = np.where(
    (~df['Size USD'].isna()) & (~df['start_position'].isna()),
    (df['Size USD'].abs()) / (df['start_position'].abs() + eps),
    np.nan
)

## Exploratory Data Analysis (EDA)

In [ ]:
OUTPUT_DIR = "/content/outputs"

# Helper: save figure and close
def save_fig(fig, fname, dpi=150):
    path = os.path.join(OUTPUT_DIR, fname)
    fig.savefig(path, bbox_inches='tight', dpi=dpi)
    plt.close(fig)
    print("Saved:", path)


In [ ]:
# 5.A Profitability vs Sentiment (mean PnL)
pnl_by_sent = df.groupby('sentiment')['closed_pnl'].agg(['mean','median','count']).reset_index()
print("\nPnL by sentiment:\n", pnl_by_sent)


PnL by sentiment:
   sentiment       mean  median  count
0      fear  89.607689     0.0  13147
1     greed  87.335448     0.0    832


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=pnl_by_sent, x='sentiment', y='mean', ax=ax)
ax.set_title('Average Closed PnL by Sentiment')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Avg Closed PnL (USD)')
save_fig(fig, 'avg_pnl_by_sentiment.png')

Saved: /content/outputs/avg_pnl_by_sentiment.png


In [ ]:
#PnL Distribution (boxplot)
fig, ax = plt.subplots(figsize=(8,5))
sns.boxplot(data=df, x='sentiment', y='closed_pnl', ax=ax)
ax.set_title('Distribution of Closed PnL by Sentiment')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Closed PnL (USD)')
ax.set_ylim(np.percentile(df['closed_pnl'].dropna(), 1), np.percentile(df['closed_pnl'].dropna(), 99))  # limit extreme outliers for visibility
save_fig(fig, 'pnl_boxplot_by_sentiment.png')
plt.show()

Saved: /content/outputs/pnl_boxplot_by_sentiment.png


In [ ]:
# Win Rate vs Sentiment
wr = df.groupby('sentiment')['win'].mean().reset_index().rename(columns={'win':'win_rate'})
wr['win_rate_pct'] = wr['win_rate'] * 100
print("\nWin rate by sentiment:\n", wr)

fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=wr, x='sentiment', y='win_rate_pct', ax=ax)
ax.set_title('Win Rate (%) by Sentiment')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Win Rate (%)')
save_fig(fig, 'winrate_by_sentiment.png')


Win rate by sentiment:
   sentiment  win_rate  win_rate_pct
0      fear  0.397353     39.735301
1     greed  0.140625     14.062500
Saved: /content/outputs/winrate_by_sentiment.png


In [ ]:
# Volume (Size USD) vs Sentiment
vol = df.groupby('sentiment')['Size USD'].agg(['sum','mean','count']).reset_index()
print("\nVolume by sentiment:\n", vol)

fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=vol, x='sentiment', y='sum', ax=ax)
ax.set_title('Total Trading Volume (USD) by Sentiment')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Total Volume (USD)')
# format y axis in millions if large
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x:,.0f}'))
save_fig(fig, 'volume_by_sentiment.png')


Volume by sentiment:
   sentiment           sum          mean  count
0      fear  4.165729e+08  31685.779195  13147
1     greed  2.336806e+06   2808.660469    832
Saved: /content/outputs/volume_by_sentiment.png


In [ ]:
# Trading activity heatmap: hour of day vs sentiment (count)
heat_df = df.groupby(['sentiment','hour']).size().reset_index(name='count')
# pivot
heat_pivot = heat_df.pivot(index='hour', columns='sentiment', values='count').fillna(0)

fig, ax = plt.subplots(figsize=(10,6))
sns.heatmap(heat_pivot, cmap='Blues', linewidths=.5, annot=False, ax=ax)
ax.set_title('Trading Activity (count) by Hour vs Sentiment')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Hour of Day')
save_fig(fig, 'activity_heatmap_hour_sentiment.png')


Saved: /content/outputs/activity_heatmap_hour_sentiment.png
